In [1]:
import re
import json
from pyvi import ViTokenizer

In [2]:
re_thuchientheo = re.compile(
    r"((((được\s)?thực hiện theo qu[iy] định tại\s|hướng dẫn tại\s|theo qu[iy] định tại\s|(được\s)?thực hiện theo\s|theo qu[iy] định tại\s|theo nội dung qu[yi] định tại\s|quy[iy] định tại|theo\s)(các\s)?)?|tại\s(các\s)?)(khoản(\ssố)?\s(\d+\,\s)*\d+|điều(\ssố)?\s(\d+\,\s)*\d+|điểm\s(([a-z]|đ)\,\s)*([a-z]|đ)\b|chương(\ssố)?\s(\d+\,\s)*\d+)((\s|\,\s|\s\,\s|\svà\s)(khoản(\ssố)?\s(\d+\,\s)*\d+|điều(\ssố)?\s(\d+\,\s)*\d+|điểm\s(([a-z]|đ)\,\s)*([a-z]|đ)\b|chương(\ssố)?\s(\d+\,\s)*\d+))*(\s(điều này|thông tư này|nghị quyết này|quyết định này|nghị định này|văn bản này|quyết định này))?"
)
re_thongtuso = re.compile(
    r"(thông tư liên tịch|thông tư|nghị quyết|quyết định|nghị định|văn bản)\s(số\s)?(([a-z0-9]|đ|\-)+\/([a-z0-9]|đ|\-|\/)*)"
)
re_ngay = re.compile(r"ngày\s\d+\/\d+\/\d+\b|ngày\s\d+tháng\d+năm\d+")
re_thang_nam = re.compile(r"tháng\s\d+\/\d+|tháng\s\d+|năm\s\d+")
re_chuong = re.compile(
    r"chương\s(iii|ii|iv|ix|viii|vii|vi|xi|xii|xiii|xiv|xix|xviii|xvii|xvi|xv|xx|v|x|i|xxiii|xxii|xxi|xxiv|xxviii|xxvii|xxvi|xxv|xxix|xxx)\b"
)
re_dieu = re.compile(r"Điều\s\d+\.")
END_PHRASES = [
    "có đúng không",
    "đúng không",
    "được không",
    "hay không",
    "được hiểu thế nào",
    "được quy định cụ thể là gì",
    "được quy định như thế nào",
    "được quy định thế nào",
    "được quy định như nào",
    "trong trường hợp như nào",
    "trong trường hợp như thế nào",
    "trong trường hợp nào",
    "trong những trường hợp nào",
    "được hiểu như thế nào",
    "được hiểu như nào",
    "như thế nào",
    "thế nào",
    "như nào",
    "là gì",
    "là ai",
    "là bao nhiêu",
    "bao nhiêu",
    "trước bao lâu",
    "là bao lâu",
    "bao lâu",
    "bao gồm gì",
    "không",
    "bao gồm những gì",
    "vào thời điểm nào",
    "gồm những giấy tờ gì",
    "những yêu cầu nào",
]

In [3]:
def remove_dieu_number(text):
    text = re_thuchientheo.sub(" ", text)
    text = re_thongtuso.sub(" ", text)
    text = re_ngay.sub(" ", text)
    text = re_thang_nam.sub(" ", text)
    text = re_chuong.sub(" ", text)
    text = re_dieu.sub(" ", text)
    return " ".join(text.split())


def remove_other_number_by_zero(text):
    for digit in ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]:
        text = text.replace(digit, "0")
    return text


def remove_punct(text):
    text = text.replace(";", ",").replace(":", ".").replace("“", " ").replace("”", " ")
    text = "".join(
        [
            c
            if c.isalpha() or c.isdigit() or c in [" ", ",", "(", ")", ".", "/", "-"]
            else " "
            for c in text
        ]
    )
    text = " ".join(text.split())
    text = text.replace("\xa0", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [4]:
def preprocess_khoan(khoan):
    khoan = khoan.lower()
    matched = re.match(r"^\d+\.(\d+\.?)?\s", khoan)  # 1. 2.2. 2.2
    if matched is not None:
        khoan = khoan[matched.span()[1]:].strip()

    else:
        matched2 = re.match(r"^[\wđ]\)\s", khoan)
        if matched2 is not None:
            khoan = khoan[matched2.span()[1]:].strip()

    khoan = remove_dieu_number(khoan)
    khoan = remove_other_number_by_zero(khoan)
    khoan = remove_punct(khoan)
    return " ".join(khoan.split())

In [5]:
def preprocess_question(q, remove_end_phrase=True):
    q = q.lower()
    q = remove_dieu_number(q)
    q = "".join([c if c.isalpha() or c.isdigit() or c == " " else " " for c in q])
    q = remove_punct(q)
    if remove_end_phrase:
        for phrase in END_PHRASES:
            if q.endswith(phrase):
                q = q[: -len(phrase)]
                break

    return q.strip()

In [6]:
def tokenize_text(text):
    return ViTokenizer.tokenize(text)


In [25]:
tokenize_text("Tôi là sinh viên Đại học Bách Khoa Hà Nội")

'Tôi là sinh_viên Đại_học Bách_Khoa Hà_Nội'

In [7]:
punc = """!"#$%&'()*+,-./:;<=>?@[\]^`{|}~"""  # noqa: W605
table = str.maketrans("", "", punc)


def clean_text(text):
    words = text.lower().split()
    result = [w.translate(table) for w in words]
    stripped = " ".join(result)
    result = " ".join(stripped.split())
    return result

In [27]:
with open("data/raw_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [28]:
data[1]

{'qid': 80037,
 'question': 'Nội dung lồng ghép vấn đề bình đẳng giới trong xây dựng văn bản quy phạm pháp luật được quy định thế nào?',
 'positive': {'cid': 151154,
  'text': 'Nội dung lồng ghép vấn đề bình đẳng giới trong xây dựng văn bản quy phạm pháp luật\nTrong phạm vi điều chỉnh của văn bản quy phạm pháp luật:\n1. Xác định nội dung liên quan đến vấn đề bình đẳng giới hoặc vấn đề bất bình đẳng giới, phân biệt đối xử về giới.\n2. Quy định các biện pháp cần thiết để thực hiện bình đẳng giới hoặc để giải quyết vấn đề bất bình đẳng giới, phân biệt đối xử về giới; dự báo tác động của các quy định đó đối với nam và nữ sau khi được ban hành.\n3. Xác định nguồn nhân lực, tài chính cần thiết để triển khai các biện pháp thực hiện bình đẳng giới hoặc để giải quyết vấn đề bất bình đẳng giới, phân biệt đối xử về giới.'},
 'negative': {'cid': 49631,
  'text': 'Vị trí, vai trò của đại biểu Quốc hội\n1. Đại biểu Quốc hội là người đại diện cho ý chí, nguyện vọng của Nhân dân ở đơn vị bầu cử ra mìn

In [20]:
import json
import pandas as pd
corpus_df = pd.read_csv("../data/full_corpus.csv")

In [21]:
def clean_corpus(data):
    clean_corpuses = []
    for i, r in data.iterrows():
        cac_khoan = r["text"].split("\n")
        khoan_clean = []
        for khoan in cac_khoan:
            khoan = preprocess_khoan(khoan)
            khoan_clean.append(khoan.strip())
        clean_text = " ".join(khoan_clean)
        clean_corpuses.append({"raw_text": r["text"], "text": clean_text, "cid": r["cid"]})
    return clean_corpuses

In [22]:
clean_corpuses = clean_corpus(corpus_df)

In [23]:
clean_corpuses[2]

{'raw_text': 'Tiêu chuẩn của các thành viên thuộc lực lượng tuần tra, canh gác đê\n1. Là người khoẻ mạnh, tháo vát, đủ khả năng đảm đương những công việc nặng nhọc, kể cả lúc mưa to, gió lớn, đêm tối.\n2. Có tinh thần trách nhiệm, chịu đựng gian khổ, khắc phục khó khăn, quen sông nước và biết bơi, có kiến thức, kinh nghiệm hộ đê, phòng, chống lụt, bão.',
 'text': 'tiêu chuẩn của các thành viên thuộc lực lượng tuần tra, canh gác đê là người khoẻ mạnh, tháo vát, đủ khả năng đảm đương những công việc nặng nhọc, kể cả lúc mưa to, gió lớn, đêm tối. có tinh thần trách nhiệm, chịu đựng gian khổ, khắc phục khó khăn, quen sông nước và biết bơi, có kiến thức, kinh nghiệm hộ đê, phòng, chống lụt, bão.',
 'cid': 2}

In [26]:
len(clean_corpuses)

261794

In [25]:
with open("../data/clean_corpus.json", "w", encoding="utf-8") as f:
    json.dump(clean_corpuses, f, ensure_ascii=False, indent=4)

In [29]:
def clean_train_data(data):
    # clean data
    for entry in data:
        entry["question"] = preprocess_question(
            entry["question"], remove_end_phrase=False
        )
        cac_khoan = entry["positive"]["text"].split("\n")
        khoan_clean = []
        for khoan in cac_khoan:
            khoan = preprocess_khoan(khoan)
            khoan_clean.append(khoan.strip())
        entry["positive"]["text"] = " ".join(khoan_clean)
        
        cac_khoan = entry["negative"]["text"].split("\n")
        khoan_clean = []
        for khoan in cac_khoan:
            khoan = preprocess_khoan(khoan)
            khoan_clean.append(khoan.strip())
        entry["negative"]["text"] = " ".join(khoan_clean)
    # tokenize data for phobert
    for entry in data:
        entry["question"] = clean_text(tokenize_text(entry["question"]))
        entry["positive"]["segmented_text"] = clean_text(tokenize_text(entry["positive"]["text"]))
        entry["negative"]["segmented_text"] = clean_text(tokenize_text(entry["negative"]["text"]))
    return data

In [30]:
data = clean_train_data(data)

In [31]:
data

[{'qid': 161615,
  'question': 'người học ngành quản_lý khai_thác công_trình thủy_lợi trình_độ cao_đẳng phải có khả_năng học_tập và nâng cao_trình_độ như thế_nào',
  'positive': {'cid': 62492,
   'text': 'khả năng học tập, nâng cao trình độ - khối lượng khối lượng kiến thức tối thiểu, yêu cầu về năng lực mà người học phải đạt được sau khi tốt nghiệp ngành, nghề mộc xây dựng và trang trí nội thất, trình độ cao đẳng có thể tiếp tục phát triển ở các trình độ cao hơn, - người học sau tốt nghiệp có năng lực tự học, tự cập nhật những tiến bộ khoa học công nghệ trong phạm vi ngành, nghề để nâng cao trình độ hoặc học liên thông lên trình độ cao hơn trong cùng ngành, nghề hoặc trong nhóm ngành, nghề hoặc trong cùng lĩnh vực đào tạo./. người học ngành mộc xây dựng và trang trí nội thất trình độ cao đẳng phải có khả năng học tập, nâng cao trình độ như thế sau. - khối lượng khối lượng kiến thức tối thiểu, yêu cầu về năng lực mà người học phải đạt được sau khi tốt nghiệp ngành, nghề mộc xây dựng và

In [32]:
len(data)

133568

In [ ]:
with open("data/triplet_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

In [8]:
with open("../data/triplet_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [5]:
data[1]

{'qid': 80037,
 'question': 'nội_dung lồng_ghép vấn_đề bình_đẳng giới trong xây_dựng văn_bản quy_phạm_pháp_luật được quy_định thế_nào',
 'positive': {'cid': 151154,
  'text': 'nội dung lồng ghép vấn đề bình đẳng giới trong xây dựng văn bản quy phạm pháp luật trong phạm vi điều chỉnh của văn bản quy phạm pháp luật. xác định nội dung liên quan đến vấn đề bình đẳng giới hoặc vấn đề bất bình đẳng giới, phân biệt đối xử về giới. quy định các biện pháp cần thiết để thực hiện bình đẳng giới hoặc để giải quyết vấn đề bất bình đẳng giới, phân biệt đối xử về giới, dự báo tác động của các quy định đó đối với nam và nữ sau khi được ban hành. xác định nguồn nhân lực, tài chính cần thiết để triển khai các biện pháp thực hiện bình đẳng giới hoặc để giải quyết vấn đề bất bình đẳng giới, phân biệt đối xử về giới.',
  'segmented_text': 'nội_dung lồng_ghép vấn_đề bình_đẳng giới trong xây_dựng văn_bản quy_phạm_pháp_luật trong phạm_vi điều_chỉnh của văn_bản quy_phạm_pháp_luật xác_định nội_dung liên_quan đế

In [6]:
classification_data = []
for entry in data:
    classification_data.append({"qid": entry["qid"], "question": entry["question"], "cid": entry["positive"]["cid"],  "context": entry["positive"]["segmented_text"], "label": 1})
    classification_data.append({"qid": entry["qid"], "question": entry["question"], "cid": entry["negative"]["cid"],  "context": entry["negative"]["segmented_text"], "label": 0})

In [7]:
len(classification_data)

267136

In [8]:
with open("../data/classification_data.json", "w", encoding="utf-8") as f:
    json.dump(classification_data, f, ensure_ascii=False, indent=4)

In [10]:
test_data = data[int(0.9*len(data)):]

In [12]:
test_data[0]

{'qid': 85845,
 'question': 'cơ_sở khảo_nghiệm giống thủy_sản vi_phạm_quy_định về khảo_nghiệm có_thể bị xử_phạt hành_chính tối_đa bao_nhiêu',
 'positive': {'cid': 12022,
  'text': 'mức phạt tiền đối với hành vi vi phạm hành chính quy định tại của nghị định này là mức phạt tiền đối với cá nhân, trừ các hành vi vi phạm hành chính quy định , , các , , , các và 00, . mức phạt tiền tối đa đối với cá nhân là 0.000.000.000 đồng. mức phạt tiền đối với cùng một hành vi vi phạm hành chính của tổ chức bằng 00 lần mức phạt tiền đối với cá nhân. mức phạt tiền tối đa đối với tổ chức là 0.000.000.000 đồng. thẩm quyền xử phạt vi phạm hành chính của những người được quy định tại các điều từ 00 đến 00 nghị định này là thẩm quyền áp dụng đối với một hành vi vi phạm hành chính của cá nhân. trong trường hợp phạt tiền, thẩm quyền xử phạt đối với tổ chức gấp 00 lần thẩm quyền xử phạt đối với cá nhân.',
  'segmented_text': 'mức phạt tiền đối_với hành_vi vi_phạm hành_chính quy_định tại của nghị_định này là mức

In [14]:
qid_set = set()
for entry in test_data:
    qid_set.add(entry["qid"])

In [15]:
len(qid_set)

11929

In [16]:
import pandas as pd
train_df = pd.read_csv("../data/train.csv")

In [17]:
test_df = train_df.tail(11929)

In [20]:
test_set = []

for i, r in test_df.iterrows():
    list_cid = r["cid"][1:-1].split()
    list_cid = [int(cid) for cid in list_cid]
    question = preprocess_question(q=r["question"], remove_end_phrase=False)
    test_set.append({"qid": r["qid"], "raw_text": r["question"], "text": question, "cid": list_cid})

In [22]:
test_set[0]

{'qid': 85845,
 'raw_text': 'Cơ sở khảo nghiệm giống thủy sản vi phạm quy định về khảo nghiệm có thể bị xử phạt hành chính tối đa bao nhiêu?',
 'text': 'cơ sở khảo nghiệm giống thủy sản vi phạm quy định về khảo nghiệm có thể bị xử phạt hành chính tối đa bao nhiêu',
 'cid': [43470, 12022]}

In [23]:
with open("../data/test_set.json", "w", encoding="utf-8") as f:
    json.dump(test_set, f, ensure_ascii=False, indent=4)